# Notebook 2 — Comparative Genomics

Analysis of the **SLC26 anion transporter family** across 30 mammalian
species.

1. **NCBI taxonomy database**
2. **Gene family tree** visualization
3. **Phylogenetic profiles**
4. **Prestin (SLC26A5)** alignment — cochlear motor protein
5. Measure **sequence conservation** with Shannon entropy

---
## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import math

sns.set_theme()

DATA = os.path.join('..', 'data')
SUB_DIR = os.path.join(DATA, 'subfamilies')

In [ ]:
def read_fasta(path):
    """Read a FASTA file → dict of {header: sequence}."""
    seqid2seq = {}
    header = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                header = line[1:].split()[0]
                seqid2seq[header] = ''
            elif header:
                seqid2seq[header] += line
    return seqid2seq

---
## 2. The NCBI Taxonomy Database

The [NCBI Taxonomy](https://www.ncbi.nlm.nih.gov/taxonomy) is a curated
classification of all organisms with sequences in GenBank. ETE4 provides
a Python interface to query it locally.

This is useful because our sequence headers contain **taxids** (taxonomy IDs)
like `9606` for *Homo sapiens* — we need to convert those into species
names, lineages, and phylogenetic relationships.

In [ ]:
from ete4 import NCBITaxa

ncbi = NCBITaxa()

### 2.1 Basic queries

In [ ]:
# Convert a taxid to a species name
taxid2name = ncbi.get_taxid_translator([9606])
print(f"Taxid 9606 -> {taxid2name[9606]}")

In [ ]:
# Get the full lineage of a species
lineage = ncbi.get_lineage(9606)
lineage_names = ncbi.get_taxid_translator(lineage)
print("Lineage of Homo sapiens:")
for tid in lineage:
    print(f"  {tid:>10d}  {lineage_names[tid]}")

In [ ]:
# Get the taxonomic rank of each level
ranks = ncbi.get_rank(lineage)
print("\nRanks:")
for tid in lineage:
    print(f"  {lineage_names[tid]:<30s}  {ranks[tid]}")

### 2.2 Topology based on NCBI taxonomy

In [ ]:
# Get the NCBI species tree for a set of taxids
some_taxids = [9606, 10090, 9739, 59479, 9913]  # human, mouse, dolphin, horseshoe bat, cow
species_tree = ncbi.get_topology(some_taxids)

# Annotate with scientific names
ncbi.annotate_tree(species_tree)

print(species_tree.to_str(props=['sci_name']))

### Exercise 1

1. Find the taxonomy of the **bottlenose dolphin** (*Tursiops truncatus*,
   taxid 9739) and the bat *Myotis lucifugus* (taxid 59463). What **order** do they belong to?

2. What is the most recent common ancestor of dolphins and bats
   according to the NCBI tree? *(build a tree with `get_topology`
   for just those two taxids and look at the root.)*

In [ ]:
# Your code here

---
## 3. The SLC26 gene family tree

We have built a tree of 297 SLC26 sequences. We load
it and annotate with taxonomy and evolutionary events.

In [ ]:
from ete4 import PhyloTree
from ete4.smartview import Layout, TextFace, RectFace

FAMILY_TREE = os.path.join(DATA, 'selection2.clustalo.gt01.fasttree.nwk')

t = PhyloTree(open(FAMILY_TREE).read(),
              sp_naming_function=lambda n: n.split('.')[0])
t.set_outgroup(t.get_midpoint_outgroup())

#t.resolve_polytomy(descendants=True)

print("Annotating with NCBI taxonomy...")
t.annotate_ncbi_taxa()

print("Inferring duplication/speciation events...")
events = t.get_descendant_evol_events()

n_dup = sum(1 for e in events if e.etype == 'D')
n_spec = sum(1 for e in events if e.etype == 'S')
print(f"Leaves: {len(list(t.leaves()))}")
print(f"Duplication events: {n_dup}")
print(f"Speciation events: {n_spec}")

### 3.1 Understanding orthology and paralogy

The tree contains two types of events:
- **Speciation** events: a gene was inherited by descendant species
  (genes related by speciation = **orthologs**)
- **Duplication** events: a gene was duplicated within a genome
  (genes related by duplication = **paralogs**)

The SLC26 family diversified through ~10 ancient duplication events, creating
subfamilies A1 through A11. Within each subfamily, all genes are orthologs, across families paralogs.

### 3.2 Loading subfamily assignments

In [ ]:
# Load gene name mapping
GNAME_FILE = os.path.join(DATA, 'selection2.clustalo.seqid2gname.tab')
seqid2gene = {}
if os.path.exists(GNAME_FILE):
    for line in open(GNAME_FILE):
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            seqid2gene[parts[0]] = parts[1]
    print(f"Loaded {len(seqid2gene)} gene name mappings")

import re

def guess_subfamily(gene_name):
    if not gene_name:
        return 'unknown'
    text = gene_name.upper()
    m = re.search(r'S(?:LC)?26A(\d+)', text)
    if m:
        return f"SLC26A{m.group(1)}"
    if 'PRESTIN' in text:
        return 'SLC26A5'
    if 'PENDRIN' in text:
        return 'SLC26A4'
    return 'unknown'

# Assign subfamily to each leaf
for leaf in t.leaves():
    gname = seqid2gene.get(leaf.name, '')
    leaf.add_prop('gene_name', gname)
    leaf.add_prop('subfamily', guess_subfamily(gname))

# Summary
sub_counts = Counter(leaf.props.get('subfamily') for leaf in t.leaves())
print(f"\n{'Subfamily':<15s} {'Count':>6s}")
print('-' * 25)
for sub, count in sorted(sub_counts.items()):
    print(f"  {sub:<15s} {count:>4d}")

### 3.3 Visualize the family tree with subfamilies colored

In [ ]:
SUBFAMILY_COLORS = {
    'SLC26A1':  '#5DCAA5',
    'SLC26A2':  '#1D9E75',
    'SLC26A3':  '#85B7EB',
    'SLC26A4':  '#F0997B',
    'SLC26A5':  '#D85A30',   # prestin — the key gene
    'SLC26A6':  '#378ADD',
    'SLC26A7':  '#97C459',
    'SLC26A8':  '#ED93B1',
    'SLC26A9':  '#AFA9EC',
    'SLC26A11': '#EF9F27',
    'unknown':  '#CCCCCC',
}

def subfamily_layout(node):
    if not node.is_leaf:
        # Mark duplication events
        if node.props.get('evoltype') == 'D':
            return {'dot': {'shape': 'circle', 'radius': 8, 'fill': 'red'}}
        return

    sub = node.props.get('subfamily', 'unknown')
    color = SUBFAMILY_COLORS.get(sub, '#CCCCCC')
    sci = node.props.get('sci_name', '')
    gname = node.props.get('gene_name', '')
    label = f"{gname} ({sci})" if gname else sci
    
    fill_color = "red" if node.props['taxid'] == 9606 else "black"

    return [
        TextFace(label, style={'fill': fill_color},
                 column=0, position='right'),
        {'box': {'fill': color}}
    ]

tree_style = {'node-height-min': 1, 'smart-zoom': False}

layouts = [
    Layout(name='TreeStyle', draw_tree=tree_style, active=True),
    Layout(name='Subfamilies', draw_node=subfamily_layout, active=True),
]

print("Launching tree viewer — look for colored subfamily clades")
print("Red squares at internal nodes = gene duplication events")
t.explore(layouts=layouts, keep_server=False, show_leaf_name=False,
          name='SLC26 family')

### Exercise 2

In the tree viewer:
1. Which subfamilies are **sister clades** (most closely related)?
2. Are there any subfamilies where some species have **more than one**
   copy? (lineage-specific duplications within a clade.)

---
## 4. Phylogenetic profile

A **phylogenetic profile** shows the number of genes per family.

### 4.1 Build the profile table

In [ ]:
# Count how many copies each species has per subfamily
profile = {}
for leaf in t.leaves():
    taxid = leaf.name.split('.')[0]
    sub = leaf.props.get('subfamily', 'unknown')
    if sub == 'unknown':
        continue
    key = (taxid, sub)
    profile[key] = profile.get(key, 0) + 1



In [ ]:
# Unique taxids and subfamilies
all_taxids = sorted(set(k[0] for k in profile))
all_subs = sorted(set(k[1] for k in profile),
                  key=lambda l: int(l.split('26A')[1]))

In [ ]:
# Build DataFrame
profile_df = pd.DataFrame(0, index=all_taxids, columns=all_subs)
for (taxid, sub), count in profile.items():
    profile_df.loc[taxid, sub] = count

# Add species names
taxid2name_map = ncbi.get_taxid_translator([int(t) for t in all_taxids])
profile_df.index = [f"{taxid2name_map.get(int(t), t)}" for t in profile_df.index]

# Keep a copy with taxids for the ETE4 visualization
profile_taxid = pd.DataFrame(0, index=all_taxids, columns=all_subs)
for (taxid, sub), count in profile.items():
    profile_taxid.loc[taxid, sub] = count

print(profile_df)

### 4.2 Visualize on the NCBI species tree

In [ ]:
# Build NCBI species tree for our taxids
species_tree = ncbi.get_topology([int(t) for t in all_taxids])
ncbi.annotate_tree(species_tree)

# Attach profile data to leaves
for leaf in species_tree.leaves():
    taxid = str(leaf.taxid)
    for sub in all_subs:
        count = int(profile_taxid.loc[taxid, sub]) if taxid in profile_taxid.index else 0
        leaf.add_prop(f'n_{sub}', count)

In [ ]:
# Layout: phylogenetic profile as colored rectangles
def profile_layout(node):
    if not node.is_leaf:
        return
    faces = []
    # One rectangle per subfamily
    for i, sub in enumerate(all_subs):
        count = node.props.get(f'n_{sub}', 0)
        if count > 0:
            color = SUBFAMILY_COLORS.get(sub, '#CCCCCC')
            label = str(count) if count > 0 else ''
            faces.append(RectFace(wmax=20, hmax=20, style={'fill': color},
                                  text=label,
                                  column=i + 1, position='aligned'))
        else:
            faces.append(RectFace(wmax=20, hmax=20, style={'fill': '#F5F5F5'},
                                  column=i + 1, position='aligned'))
    return faces

# Header: subfamily names
def profile_header(tree):
    faces = []
    for i, sub in enumerate(all_subs):
        color = SUBFAMILY_COLORS.get(sub, '#CCCCCC')
        short = sub.replace('SLC26', '')
        faces.append(TextFace(short, column=i + 1, position='header',
                              rotation=-90,
                              style={'fill': color, 'font-size': '10px'}))
    return faces

def node_names_style(node):
    if node.is_leaf:
        acc = node.name.split(".")[1] if "." in node.name else node.name
        sci = node.props.get("sci_name", "")
        return [
            TextFace(f"{sci}", style={"fill": "gray"}, column=0, position="right"),
        ]

profile_tree_style = {'node-height-min': 15, 'smart-zoom': False}

profile_layouts = [
    Layout(name="Names", draw_node=node_names_style, active=True),
    Layout(name='ProfileStyle', draw_tree=profile_tree_style, active=True),
    Layout(name='Profile', draw_node=profile_layout, draw_tree=profile_header, active=True),
]

species_tree.to_ultrametric()

print("Launching phylogenetic profile viewer...")
species_tree.explore(layouts=profile_layouts, keep_server=False,
                     show_leaf_name=False, name='SLC26 phylogenetic profile')

### Exercise 3

From the phylogenetic profile:
1. Do all species have all subfamilies? Which species have **fewer**
   genes than expected?
2. Is there any subfamily that's missing from multiple species?

---
## 5. The prestin (SLC26A5) sub-alignment

From the family tree, we extracted the prestin clade — one ortholog
per species — and re-aligned them with MAFFT, then trimmed. We'll
visualize the alignment and the prestin tree.

### 5.1 Visualize with pyMSAviz

In [ ]:
from pymsaviz import MsaViz

# Use the heavily trimmed alignment for cleaner visualization
PRESTIN_VIZ = os.path.join(SUB_DIR, 'SLC26A5.trim099.fa')
if not os.path.exists(PRESTIN_VIZ):
    # Fall back to lighter trimming
    PRESTIN_VIZ = os.path.join(SUB_DIR, 'SLC26A5.trim.fa')
    print(f"Using -gt 0.1 trimmed alignment: {PRESTIN_VIZ}")
else:
    print(f"Using -gt 0.99 trimmed alignment: {PRESTIN_VIZ}")

mv = MsaViz(PRESTIN_VIZ, wrap_length=80, show_count=True)
fig = mv.plotfig()
plt.show()

### 5.2 Prestin tree with echolocators highlighted

In [ ]:
# Load species classification
species_df = pd.read_csv(
    os.path.join(DATA, 'species_classification.tsv'),
    sep='\t', comment='#',
    names=['taxid', 'species', 'group', 'echolocates', 'notes']
)
species_df['taxid'] = species_df['taxid'].astype(str)

echo_lookup = dict(zip(species_df['taxid'], species_df['echolocates']))
name_lookup = dict(zip(species_df['taxid'], species_df['species']))
group_lookup = dict(zip(species_df['taxid'], species_df['group']))

In [ ]:
PRESTIN_TREE = os.path.join(SUB_DIR, 'SLC26A5.nwk')
pt = PhyloTree(open(PRESTIN_TREE).read(),
               sp_naming_function=lambda n: n.split('.')[0])
# pt.set_outgroup(pt.get_midpoint_outgroup())
pt.set_outgroup(pt.common_ancestor(['13616.F6TKX6', '38626.A0A6P5K5E7']))

pt.annotate_ncbi_taxa()

ECHO_COLORS = {
    'echolocating_bat':         '#D85A30',
    'echolocating_cetacean':    '#1D9E75',
    'non_echolocating_bat':     '#7F77DD',
    'non_echolocating_cetacean':'#378ADD',
    'outgroup':                 '#888888',
}

def prestin_echo_layout(node):
    if not node.is_leaf:
        return
    taxid = node.name.split('.')[0]
    sci = node.props.get('sci_name', taxid)
    group = group_lookup.get(taxid, 'outgroup')
    echo = echo_lookup.get(taxid, 'no')
    color = ECHO_COLORS.get(group, '#888888')
    marker = '  ᯤ' if echo == 'yes' else ''

    return [
        TextFace(f"{sci}{marker}",
                 style={'fill': color,
                        'font-weight': 'bold' if echo == 'yes' else 'normal'},
                 column=0, position='right'),
    ]

pt_style = {'node-height-min': 1, 'smart-zoom': False}

pt.explore(
    layouts=[
        Layout(name='Style', draw_tree=pt_style),
        Layout(name='Echolocation', draw_node=prestin_echo_layout),
    ],
    keep_server=False, show_leaf_name=False, name='Prestin tree'
)

### Exercise 4

In the prestin tree:
1. Do the echolocating species (coral = bats, teal = dolphins) form
   a **single clade**, or are they scattered across the tree?
2. Does the tree reflect the actual **evolutionary relationships** of the species (e.g., are
   all bats together, all cetaceans together)?
3. What does this tell you about convergent evolution at the **tree
   topology** level vs. at the **individual amino acid** level?

---
## 6. Working with alignment data in pandas

Convert the prestin alignment into a pandas DataFrame to
analyze it as a table.

In [ ]:
# Load the analysis-level alignment (lighter trim)
PRESTIN_ALN = os.path.join(SUB_DIR, 'SLC26A5.trim.fa')
prestin = read_fasta(PRESTIN_ALN)

print(f"Sequences:        {len(prestin)}")
print(f"Alignment length: {len(list(prestin.values())[0])} positions")

### 6.1 Creating a DataFrame from the alignment

In [ ]:
headers = list(prestin.keys())
matrix = [list(seq) for seq in prestin.values()]

aln = pd.DataFrame(matrix, index=headers)
aln.columns.name = 'position'

print(f"Shape: {aln.shape}")
print(f"{aln.shape[0]} rows (sequences / species)")
print(f"{aln.shape[1]} columns (alignment positions)")

In [ ]:
aln.head(3)

### 6.2 Selecting data in pandas

In [ ]:
# Select a single column → one alignment position
# This returns a "Series" — like a labeled list
aln[0]

In [ ]:
# Count amino acids at position 0
aln[0].value_counts()

In [ ]:
# Select a range of columns
aln[[50, 51, 52, 53, 54]]

In [ ]:
# Select a single row by position number (0-based)
aln.iloc[0]

In [ ]:
# Select a row by its label (header name)
first_header = aln.index[0]
aln.loc[first_header]

### 6.3 Boolean indexing — selecting based on condition

Select rows where a condition is True.

In [ ]:
# Build a True/False series: is this species an echolocator?
taxids = [h.split('.')[0] for h in aln.index]
is_echo = pd.Series(
    [echo_lookup.get(t) == 'yes' for t in taxids],
    index=aln.index
)

print(f"Echolocating species:\t\t{is_echo.sum()}")
print(f"Non-echolocating species:\t{(~is_echo).sum()}")

In [ ]:
is_echo

In [ ]:
# Select only echolocator rows
echo_aln = aln[is_echo]
echo_aln.head()

In [ ]:
# The tilde "~"" is NOT : non-echolocators
other_aln = aln[~is_echo]
other_aln.head()

### Exercise 5

1. Pick a few alignment positions and compare amino acids between
   echolocators and non-echolocators using `value_counts()`.
2. Can you find a position where echolocators **mostly agree** on
   one amino acid, but non-echolocators are more diverse?
3. How would you compute the **gap fraction** at a given position?
   *(Hint: count how many entries equal `'-'`, divide by total.)*

In [ ]:
# Your code here

---
## 7. Shannon entropy — measuring conservation

We want a single number that captures how "conserved" an alignment
position is.

### 7.1 What should "conservation" mean?

Look at these three alignment columns (4 species each):

| Column | Species 1 | Species 2 | Species 3 | Species 4 |
|--------|-----------|-----------|-----------|-----------|
| A | Ala | Ala | Ala | Ala |
| B | Ala | Gly | Ala | Gly |
| C | Ala | Gly | Leu | Lys |

### Exercise 6a

**Rank** columns A, B, C from most conserved to least conserved.

### 7.2 [Shannon entropy](https://en.wikipedia.org/wiki/Entropy_(information_theory)) (information theory)

**Shannon entropy** measures how
"unpredictable" a position is.

$$H = -\sum_{i=1}^{k} p_i \cdot \log_2(p_i)$$

where $p_i$ is the frequency of amino acid $i$, and $k$ is the
number of distinct amino acids.

- Perfectly conserved (one amino acid): $H = 0$
- Two amino acids at 50/50: $H = 1.0$
- All 20 amino acids equally likely: $H = \log_2(20) \approx 4.3$

Low entropy = conserved. High entropy = variable.

### 7.3 Computing entropy

In [ ]:
# We compute entropy at position X
col = list(aln[100])

# Remove gaps
residues = [aa for aa in col if aa != '-']
print(f"Position 100: {len(residues)} residues (no gaps)")

# Count each amino acid
counts = Counter(residues)
total = len(residues)
print(f"Counts: {dict(counts)}")
print()

# Compute frequencies and entropy contributions
entropy = 0.0
for aa, count in counts.items():
    p = count / total
    contribution = -p * math.log2(p)
    entropy += contribution
    print(f"{aa}: p = {count}/{total} = {p:.3f}"
          f"->  -p·log₂(p) = {contribution:.4f}")

print(f"\nShannon entropy at position 100: {entropy:.3f}")

### Exercise 6b

**Write a function** that computes Shannon entropy for a list
of amino acids (ignoring gaps):

```python
def shannon_entropy(column):
    """Shannon entropy of a list of amino acids (ignoring gaps)."""
    # 1. Remove gaps
    # 2. Count each amino acid
    # 3. Compute frequencies
    # 4. Apply the formula: H = -Σ p·log₂(p)
    # 5. Return H
    pass
```

Test your function:
- All same amino acid - should return 0.0
- Half A, half G - should return 1.0
- Position 100 - should match the calculation above

In [ ]:
def shannon_entropy(column):
    """Shannon entropy of a list of amino acids (ignoring gaps)."""
    # YOUR CODE
    pass

In [ ]:
# Test
# print(shannon_entropy(['A', 'A', 'A', 'A']))          # 0.0
# print(shannon_entropy(['A', 'G', 'A', 'G']))          # 1.0
# print(shannon_entropy([aln[100]])                 # should match above

### 7.4 Conservation profile across the whole alignment

In [ ]:
n_pos = aln.shape[1]
entropies = [shannon_entropy(list(aln[pos])) for pos in range(n_pos)]

print(f"Computed entropy for {n_pos} positions")
print(f"Min: {min(entropies):.3f}  (most conserved)")
print(f"Max: {max(entropies):.3f}  (most variable)")
print(f"Mean: {np.mean(entropies):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(range(n_pos), entropies, alpha=0.4, color='steelblue')
ax.plot(range(n_pos), entropies, linewidth=0.5, color='steelblue')
ax.set_xlabel('Alignment position')
ax.set_ylabel('Shannon entropy')
ax.set_title('Prestin (SLC26A5) — conservation profile')
ax.set_xlim(0, n_pos)
plt.tight_layout()
plt.show()

Lows = conserved regions.  
Peaks = variable regions.

### Exercise 7

1. How many positions have entropy = 0 (perfectly conserved)?
   What fraction of the alignment is that?

2. What are the 10 **most conserved** positions with non-zero entropy?
   What amino acids are at those positions?

3. What are the 10 **most variable** positions? Are they near the
   beginning/end of the alignment (where trimming may be imperfect)?

4. Compute the **mean entropy** separately for echolocators and
   non-echolocators. Is the prestin protein more or less conserved
   in echolocators?

In [ ]:
# Your code here

---
## 8. Prestin 3D structure

Human prestin has a cryo-EM structure: **PDB 7S8X** (Bavi et al.
2021). We can visualize it using PyMOL's Python API, which renders
directly to an image inside the notebook.

The protein has three main regions:
- **TMD** (transmembrane domain, ~residues 81–505) — the motor domain
- **Linker** (~506–530)
- **STAS domain** (~531–744) — cytoplasmic regulatory domain

See also: https://www.uniprot.org/uniprotkb/P58743/entry#structure

### Installation

```bash
conda install conda-forge::pymol-open-source
```

In [ ]:
try:
    import pymol2
    from IPython.display import Image, display
    import tempfile

    with pymol2.PyMOL() as p:
        p.cmd.load('7s8x.cif', 'prestin')
        p.cmd.remove('solvent')
        p.cmd.remove('chain B')

        p.cmd.hide('everything')
        p.cmd.show('cartoon', 'prestin')
        p.cmd.set('cartoon_transparency', 0.1)

        # Color by domain
        p.cmd.color('gray80', 'prestin')
        p.cmd.color('tv_red', 'prestin and resi 81-505')      # TMD
        p.cmd.color('tv_orange', 'prestin and resi 506-530')   # Linker
        p.cmd.color('tv_blue', 'prestin and resi 531-744')     # STAS

        p.cmd.bg_color('white')
        p.cmd.orient('prestin')
        p.cmd.set('ray_opaque_background', 0)

        img_path = os.path.join(tempfile.gettempdir(), 'prestin_domains.png')
        p.cmd.png(img_path, width=900, height=600, ray=1)

    display(Image(img_path))
    print("Red = TMD (motor domain) | Orange = linker | Blue = STAS")

except ImportError:
    print("PyMOL not installed — run: conda install conda-forge::pymol-open-source")
    print("Explore the structure at: https://www.rcsb.org/3d-view/7S8X")

The TMD (red) is where prestin's motor function lies. If
convergent evolution echolocation functionality, we may
expect the TMD to be enriched in convergent sites.

---
## Main points

1. **NCBI taxonomy database** through ETE4
2. SLC26 **gene family tree** with colored subfamilies
3. **Phylogenetic profile** - gene presence/absence
4. **Pandas** — DataFrames, slicing, boolean indexing
5. **Shannon entropy** - conservation
6. **3D structure** with PyMOL